In [3]:
import cv2
import os
import time

# =========================
# KONFIGURASI
# =========================
DATASET_PATH = "Percobaan_bigru/dataset_kata_video"

LABELS = [
    "berasal",
    "berpikir",
    "maaf",
    "makan",
    "mandi",
    "nama",
    "perkenalkan",
    "saya",
    "tidur",
    "tolong",
    "tunggu",
    "umur",
    "berteman",
    "sementara",
    "santai aja",
    "bareng",
]

VIDEOS_PER_LABEL = 30
VIDEO_DURATION   = 3      # durasi rekam tiap video (detik)
FPS              = 20

COUNTDOWN     = 3         # countdown sebelum video pertama tiap label
CAMERA_INDEX  = 0

# =========================
# SETUP DATASET
# =========================
os.makedirs(DATASET_PATH, exist_ok=True)

for label in LABELS:
    os.makedirs(os.path.join(DATASET_PATH, label), exist_ok=True)

# =========================
# BUKA KAMERA
# =========================
cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    raise RuntimeError("Kamera tidak terbuka. Coba ganti CAMERA_INDEX ke 1.")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

stop_program = False

print("Kontrol:")
print("Mode OTOMATIS per kata — video dalam 1 kata direkam beruntun tanpa tekan S")
print("SPASI = lanjut ke kata berikutnya (setelah satu kata selesai)")
print("N = skip kata ini lebih awal (sebelum 50 video selesai)")
print("P = pause / lanjut (jeda sementara)")
print("Q = keluar")


# =========================
# FUNGSI TAMPILKAN TEKS
# =========================
def put_text(frame, text, y, color=(255, 255, 255), scale=0.8, thickness=2):
    cv2.putText(
        frame,
        text,
        (30, y),
        cv2.FONT_HERSHEY_SIMPLEX,
        scale,
        color,
        thickness
    )


def show_and_check_key(frame, window="Record Dataset Kata"):
    """Tampilkan frame, return key yang ditekan (atau None)."""
    cv2.imshow(window, frame)
    key = cv2.waitKey(1) & 0xFF
    return key


# =========================
# LOOP UTAMA
# =========================
try:
    for label in LABELS:
        if stop_program:
            break

        print(f"\n=== Mulai merekam kata '{label}' (otomatis) ===")

        label_path = os.path.join(DATASET_PATH, label)

        existing_videos = [
            f for f in os.listdir(label_path)
            if f.lower().endswith((".mp4", ".avi", ".mov"))
        ]
        count = len(existing_videos)

        skip_label = False
        paused = False

        # ---- COUNTDOWN AWAL SEBELUM LABEL INI DIMULAI ----
        for sec in range(COUNTDOWN, 0, -1):
            start_c = time.time()
            while time.time() - start_c < 1:
                ret, frame = cap.read()
                if not ret:
                    stop_program = True
                    break

                frame = cv2.flip(frame, 1)
                disp = frame.copy()
                put_text(disp, f"Siap kata: {label}", 40, (0, 255, 0), 1)
                put_text(disp, f"Mulai dalam {sec}...", 90, (0, 0, 255), 1.2)
                put_text(disp, "Posisikan tangan dari awal gerakan", 140, (0, 255, 255), 0.7)

                key = show_and_check_key(disp)
                if key == ord("q"):
                    stop_program = True
                    break

            if stop_program:
                break

        if stop_program:
            break

        # ---- LOOP REKAM OTOMATIS UNTUK LABEL INI ----
        while count < VIDEOS_PER_LABEL and not skip_label and not stop_program:

            # ---- CEK PAUSE SEBELUM REKAM ----
            while paused:
                ret, frame = cap.read()
                if not ret:
                    stop_program = True
                    break

                frame = cv2.flip(frame, 1)
                disp = frame.copy()
                put_text(disp, "PAUSE — tekan P untuk lanjut", 40, (0, 255, 255), 1)
                put_text(disp, f"Kata: {label} | Video: {count + 1}/{VIDEOS_PER_LABEL}", 80)

                key = show_and_check_key(disp)
                if key == ord("p"):
                    paused = False
                elif key == ord("q"):
                    stop_program = True
                    break
                elif key == ord("n"):
                    skip_label = True
                    paused = False
                    break

            if stop_program or skip_label:
                break

            # ---- REKAM VIDEO ----
            filename = os.path.join(
                label_path,
                f"{label}_{count + 1:03d}.mp4"
            )

            out = cv2.VideoWriter(filename, fourcc, FPS, (frame_width, frame_height))

            print(f"Merekam otomatis: {filename}")

            start_time = time.time()

            while time.time() - start_time < VIDEO_DURATION:
                ret, frame = cap.read()
                if not ret:
                    print("Frame kamera tidak terbaca saat rekam.")
                    stop_program = True
                    break

                frame = cv2.flip(frame, 1)

                elapsed = time.time() - start_time
                remaining = max(0, VIDEO_DURATION - elapsed)

                record_frame = frame.copy()
                put_text(record_frame, f"REC otomatis: {label}", 40, (0, 0, 255), 1)
                put_text(record_frame, f"Sisa: {remaining:.1f} detik", 80, (0, 0, 255), 0.8)
                put_text(record_frame, f"Video: {count + 1}/{VIDEOS_PER_LABEL}", 120, (255, 255, 255), 0.7)
                put_text(record_frame, "N: skip | P: pause | Q: keluar", 160, (0, 255, 255), 0.6)

                out.write(frame)

                key = show_and_check_key(record_frame)
                if key == ord("q"):
                    stop_program = True
                    break
                elif key == ord("n"):
                    skip_label = True
                    break
                elif key == ord("p"):
                    paused = True

            out.release()

            if stop_program:
                break

            if skip_label:
                print(f"Skip kata '{label}' (video terakhir dibatalkan jika belum selesai).")
                # Hapus file jika rekaman dibatalkan di tengah jalan
                if os.path.exists(filename) and time.time() - start_time < VIDEO_DURATION - 0.3:
                    os.remove(filename)
                break

            count += 1
            print(f"Selesai video {count}/{VIDEOS_PER_LABEL} untuk kata '{label}'")
            # Lanjut langsung ke video berikutnya dalam kata yang sama (tanpa jeda durasi)

        print(f"Label kata '{label}' selesai. Total video: {count}/{VIDEOS_PER_LABEL}")

        # ---- TUNGGU KONFIRMASI MANUAL SEBELUM LANJUT KE KATA BERIKUTNYA ----
        if not stop_program and label != LABELS[-1]:
            next_label = LABELS[LABELS.index(label) + 1]

            while True:
                ret, frame = cap.read()
                if not ret:
                    stop_program = True
                    break

                frame = cv2.flip(frame, 1)
                disp = frame.copy()
                put_text(disp, f"Kata '{label}' selesai! ({count} video)", 40, (0, 255, 0), 1)
                put_text(disp, f"Tekan SPASI untuk lanjut ke kata '{next_label}'", 90, (255, 200, 0), 0.9)
                put_text(disp, "Q: keluar", 130, (0, 255, 255), 0.7)

                key = show_and_check_key(disp)
                if key == ord(" "):
                    break
                elif key == ord("q"):
                    stop_program = True
                    break

finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Program selesai dengan aman.")

Kontrol:
Mode OTOMATIS per kata — video dalam 1 kata direkam beruntun tanpa tekan S
SPASI = lanjut ke kata berikutnya (setelah satu kata selesai)
N = skip kata ini lebih awal (sebelum 50 video selesai)
P = pause / lanjut (jeda sementara)
Q = keluar

=== Mulai merekam kata 'berasal' (otomatis) ===
Label kata 'berasal' selesai. Total video: 31/30

=== Mulai merekam kata 'berpikir' (otomatis) ===
Label kata 'berpikir' selesai. Total video: 30/30

=== Mulai merekam kata 'maaf' (otomatis) ===
Label kata 'maaf' selesai. Total video: 30/30

=== Mulai merekam kata 'makan' (otomatis) ===
Label kata 'makan' selesai. Total video: 30/30

=== Mulai merekam kata 'mandi' (otomatis) ===
Label kata 'mandi' selesai. Total video: 30/30

=== Mulai merekam kata 'nama' (otomatis) ===
Label kata 'nama' selesai. Total video: 30/30

=== Mulai merekam kata 'perkenalkan' (otomatis) ===
Label kata 'perkenalkan' selesai. Total video: 30/30

=== Mulai merekam kata 'saya' (otomatis) ===
Label kata 'saya' selesai. To

In [1]:
# Ambil Kata Biasa
import cv2
import mediapipe as mp
import os
import time

# ==========================================================
# KONFIGURASI - UBAH SESUAI KEBUTUHAN
# ==========================================================
SAVE_DIR     = "data_kata5_old"   # folder penyimpanan
TOTAL_GAMBAR = 50               # jumlah foto per label
COUNTDOWN    = 3                # hitung mundur sebelum mulai (detik)
DELAY_ANTAR  = 0.3              # jeda antar foto (detik)

LABEL_KATA = [
    "-",
    "salam kenal",
    "berasal",
    "berpikir",
    "makan",
    "mandi",
    "saya",
    "tidur",
    "nama",
    "maaf",
    "tolong",
]

# ==========================================================
# MEDIAPIPE TANGAN
# ==========================================================
mp_hands = mp.solutions.hands
hands    = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

def draw_hand(frame, hand_lm):
    h, w, _ = frame.shape
    pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand_lm.landmark]
    for c in mp_hands.HAND_CONNECTIONS:
        cv2.line(frame, pts[c[0]], pts[c[1]], (255, 255, 255), 2)
    for pt in pts:
        cv2.circle(frame, pt, 5, (0, 0, 255), -1)

def panel(frame, x, y, w, h, alpha=0.55):
    ov = frame.copy()
    cv2.rectangle(ov, (x, y), (x+w, y+h), (0, 0, 0), -1)
    cv2.addWeighted(ov, alpha, frame, 1-alpha, 0, frame)

# ==========================================================
# BUAT FOLDER DATASET
# ==========================================================
os.makedirs(SAVE_DIR, exist_ok=True)
for lb in LABEL_KATA:
    os.makedirs(os.path.join(SAVE_DIR, lb), exist_ok=True)

# ==========================================================
# KAMERA
# ==========================================================
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  960)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 540)

label_idx       = 0
state           = "MENU"
count           = 0
countdown_start = 0
last_capture    = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame  = cv2.flip(frame, 1)
    H, W   = frame.shape[:2]
    rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    # Landmark tangan
    if result.multi_hand_landmarks:
        for hl in result.multi_hand_landmarks:
            draw_hand(frame, hl)

    label_now = LABEL_KATA[label_idx]

    # ======================================================
    # MENU
    # ======================================================
    if state == "MENU":
        panel(frame, 0, 0, W, 110)
        cv2.putText(frame, "AMBIL DATASET KATA",
                    (15, 38), cv2.FONT_HERSHEY_DUPLEX, 1.0, (0, 255, 150), 2)
        cv2.putText(frame, f"Label aktif : {label_now.upper()}",
                    (15, 75), cv2.FONT_HERSHEY_DUPLEX, 0.85, (0, 220, 255), 2)
        ex = len([f for f in os.listdir(os.path.join(SAVE_DIR, label_now)) if f.endswith(".jpg")])
        cv2.putText(frame, f"Foto tersimpan : {ex} / {TOTAL_GAMBAR}",
                    (15, 103), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)

        # Daftar label bawah (2 kolom supaya 10 label muat)
        panel(frame, 0, H-175, W, 175)
        cv2.putText(frame, "[1-9,0] Pilih Label    [SPACE] Mulai    [Q] Keluar",
                    (15, H-152), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255, 255, 100), 1)
        for i, lb in enumerate(LABEL_KATA):

            warna  = (0, 255, 150) if i == label_idx else (180, 180, 180)
            prefix = ">>" if i == label_idx else "  "
        
            ex2 = len([
                f for f in os.listdir(os.path.join(SAVE_DIR, lb))
                if f.endswith(".jpg")
            ])
        
            bar = "X" * int(ex2 / TOTAL_GAMBAR * 10)
        
            # Tombol keyboard
            if i < 9:
                key_ch = str(i + 1)
            elif i == 9:
                key_ch = "0"
            else:
                key_ch = "-"
        
            teks = f"{prefix} [{key_ch}] {lb:<14} {ex2:>2}/{TOTAL_GAMBAR} {bar}"
        
            # 6 kiri, sisanya kanan
            if i < 6:
                col = 15
                row = H - 128 + (i * 20)
            else:
                col = W // 2
                row = H - 128 + ((i - 6) * 20)
        
            cv2.putText(
                frame,
                teks,
                (col, row),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.52,
                warna,
                1
            )
            cv2.putText(frame, teks, (col, row),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.52, warna, 1)

    # ======================================================
    # COUNTDOWN
    # ======================================================
    elif state == "COUNTDOWN":
        elapsed = time.time() - countdown_start
        sisa    = COUNTDOWN - int(elapsed)

        panel(frame, 0, 0, W, 105)
        cv2.putText(frame, f"Berpose untuk : {label_now.upper()}",
                    (15, 42), cv2.FONT_HERSHEY_DUPLEX, 0.9, (0, 220, 255), 2)

        if sisa > 0:
            cv2.putText(frame, f"Mulai dalam  {sisa} ...",
                        (15, 82), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 100), 2)
        else:
            state        = "CAPTURING"
            count        = 0
            last_capture = time.time()

    # ======================================================
    # CAPTURING
    # ======================================================
    elif state == "CAPTURING":
        now = time.time()

        if now - last_capture >= DELAY_ANTAR:

            berhasil = False
        
            # Label kosong
            if label_now == "-":
        
                if result.multi_hand_landmarks is None:
        
                    fname = os.path.join(
                        SAVE_DIR,
                        label_now,
                        f"kosong_{count+1:03d}.jpg"
                    )
        
                    cv2.imwrite(fname, frame)
                    berhasil = True
        
            # Label gesture biasa
            else:
        
                if result.multi_hand_landmarks:
        
                    fname = os.path.join(
                        SAVE_DIR,
                        label_now,
                        f"{label_now}_{count+1:03d}.jpg"
                    )
        
                    cv2.imwrite(fname, frame)
                    berhasil = True
        
            # Tambah count hanya jika berhasil simpan
            if berhasil:
                count += 1
                last_capture = now
        
                print(f"Tersimpan [{label_now}] {count}/{TOTAL_GAMBAR}")

        if count >= TOTAL_GAMBAR:
            state = "DONE"

        panel(frame, 0, 0, W, 105)
        cv2.putText(frame, f"Merekam : {label_now.upper()}",
                    (15, 42), cv2.FONT_HERSHEY_DUPLEX, 0.9, (0, 255, 100), 2)
        cv2.putText(frame, f"Foto : {count} / {TOTAL_GAMBAR}",
                    (15, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 220, 255), 2)

        # Progress bar
        bx, by, bw, bh = 15, 90, W-30, 10
        cv2.rectangle(frame, (bx, by), (bx+bw, by+bh), (50, 50, 50), -1)
        fill = int(bw * count / TOTAL_GAMBAR)
        if fill > 0:
            cv2.rectangle(frame, (bx, by), (bx+fill, by+bh), (0, 220, 120), -1)
        cv2.rectangle(frame, (bx, by), (bx+bw, by+bh), (150, 150, 150), 1)

        # Indikator REC kedip
        if int(now * 3) % 2 == 0:
            cv2.circle(frame, (W-30, 30), 10, (0, 0, 255), -1)
            cv2.putText(frame, "REC", (W-68, 37),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 2)

    # ======================================================
    # DONE
    # ======================================================
    elif state == "DONE":
        panel(frame, W//2-220, H//2-70, 440, 140)
        cv2.putText(frame, "SELESAI !",
                    (W//2-90, H//2-25), cv2.FONT_HERSHEY_DUPLEX, 1.3, (0, 255, 150), 3)
        cv2.putText(frame, f"{TOTAL_GAMBAR} foto [{label_now}] tersimpan",
                    (W//2-195, H//2+15), cv2.FONT_HERSHEY_SIMPLEX, 0.68, (200, 200, 200), 1)
        cv2.putText(frame, "SPACE = label berikutnya   Q = keluar",
                    (W//2-195, H//2+48), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255, 255, 100), 1)

    cv2.imshow("Ambil Dataset Kata", frame)

    # ======================================================
    # KEYBOARD
    # ======================================================
    key = cv2.waitKey(1) & 0xFF

    if key == ord('q'):
        break

    # Tombol 1-9 untuk label 1-9, tombol 0 untuk label ke-10
    key_map = {ord(str(i+1)): i for i in range(9)}
    key_map[ord("0")] = 9
    for k, idx in key_map.items():
        if key == k and idx < len(LABEL_KATA):
            label_idx = idx
            state     = "MENU"

    if key == ord(' '):
        if state == "MENU":
            ex = len([f for f in os.listdir(os.path.join(SAVE_DIR, label_now)) if f.endswith(".jpg")])
            if ex >= TOTAL_GAMBAR:
                print(f"  [{label_now}] sudah penuh ({ex} foto), pindah label berikutnya.")
                if label_idx < len(LABEL_KATA) - 1:
                    label_idx += 1
            else:
                state           = "COUNTDOWN"
                countdown_start = time.time()

        elif state == "DONE":
            if label_idx < len(LABEL_KATA) - 1:
                label_idx += 1
                state      = "MENU"
            else:
                print("\n Semua label selesai!")
                break

cap.release()
cv2.destroyAllWindows()
print(f"\n Dataset tersimpan di : {os.path.abspath(SAVE_DIR)}/")

Program ditutup...

 Dataset tersimpan di : D:\SEMESTER 6\Praktikum Computer Vision\ProgramPCV\data_kata5_old/


In [1]:
import cv2
import mediapipe as mp
import os
import time

# ==========================================================
# KONFIGURASI - UBAH SESUAI KEBUTUHAN
# ==========================================================
SAVE_DIR     = "data_kata6"   # folder penyimpanan
TOTAL_GAMBAR = 70               # jumlah foto per label
COUNTDOWN    = 3                # hitung mundur sebelum mulai (detik)
DELAY_ANTAR  = 0.3              # jeda antar foto (detik)

LABEL_KATA = [
    "salam kenal",
    "berasal",
    "berpikir",
    "makan",
    "mandi",
    "saya",
    "tidur",
    "nama",
    "maaf",
    "tolong",
    "mulai",
    "sementara",
    "tunggu",
    "bantu",
    "berteman",
    "sehat",
    "masalah"
]

# ==========================================================
# MEDIAPIPE TANGAN
# ==========================================================
mp_hands = mp.solutions.hands
hands    = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

def draw_hand(frame, hand_lm):
    h, w, _ = frame.shape
    pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand_lm.landmark]
    for c in mp_hands.HAND_CONNECTIONS:
        cv2.line(frame, pts[c[0]], pts[c[1]], (255, 255, 255), 2)
    for pt in pts:
        cv2.circle(frame, pt, 5, (0, 0, 255), -1)

def panel(frame, x, y, w, h, alpha=0.55):
    ov = frame.copy()
    cv2.rectangle(ov, (x, y), (x+w, y+h), (0, 0, 0), -1)
    cv2.addWeighted(ov, alpha, frame, 1-alpha, 0, frame)

# ==========================================================
# BUAT FOLDER DATASET
# ==========================================================
os.makedirs(SAVE_DIR, exist_ok=True)
for lb in LABEL_KATA:
    os.makedirs(os.path.join(SAVE_DIR, lb), exist_ok=True)

# ==========================================================
# KAMERA
# ==========================================================
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  960)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 540)

label_idx       = 0
state           = "MENU"
count           = 0
countdown_start = 0
last_capture    = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame  = cv2.flip(frame, 1)
    H, W   = frame.shape[:2]
    rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    # Landmark tangan
    if result.multi_hand_landmarks:
        for hl in result.multi_hand_landmarks:
            draw_hand(frame, hl)

    label_now = LABEL_KATA[label_idx]

    # ======================================================
    # MENU
    # ======================================================
    if state == "MENU":
        panel(frame, 0, 0, W, 110)
        cv2.putText(frame, "AMBIL DATASET KATA",
                    (15, 38), cv2.FONT_HERSHEY_DUPLEX, 1.0, (0, 255, 150), 2)
        cv2.putText(frame, f"Label aktif : {label_now.upper()}",
                    (15, 75), cv2.FONT_HERSHEY_DUPLEX, 0.85, (0, 220, 255), 2)
        ex = len([f for f in os.listdir(os.path.join(SAVE_DIR, label_now)) if f.endswith(".jpg")])
        cv2.putText(frame, f"Foto tersimpan : {ex} / {TOTAL_GAMBAR}",
                    (15, 103), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)

        # Daftar label bawah (2 kolom supaya 10 label muat)
        panel(frame, 0, H-175, W, 175)
        cv2.putText(frame, "[1-9,0] Pilih Label    [SPACE] Mulai    [Q] Keluar",
                    (15, H-152), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255, 255, 100), 1)
        for i, lb in enumerate(LABEL_KATA):

            warna  = (0, 255, 150) if i == label_idx else (180, 180, 180)
            prefix = ">>" if i == label_idx else "  "
        
            ex2 = len([
                f for f in os.listdir(os.path.join(SAVE_DIR, lb))
                if f.endswith(".jpg")
            ])
        
            bar = "X" * int(ex2 / TOTAL_GAMBAR * 10)
        
            teks = f"{prefix} {lb:<15} {ex2:>2}/{TOTAL_GAMBAR}"
        
            # =========================
            # AUTO GRID
            # =========================
            per_col = 7   # jumlah baris per kolom
        
            col_idx = i // per_col
            row_idx = i % per_col
        
            col = 15 + (col_idx * 310)
            row = H - 145 + (row_idx * 22)
        
            cv2.putText(
                frame,
                teks,
                (col, row),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.52,
                warna,
                1
            )

    # ======================================================
    # COUNTDOWN
    # ======================================================
    elif state == "COUNTDOWN":
        elapsed = time.time() - countdown_start
        sisa    = COUNTDOWN - int(elapsed)

        panel(frame, 0, 0, W, 105)
        cv2.putText(frame, f"Berpose untuk : {label_now.upper()}",
                    (15, 42), cv2.FONT_HERSHEY_DUPLEX, 0.9, (0, 220, 255), 2)

        if sisa > 0:
            cv2.putText(frame, f"Mulai dalam  {sisa} ...",
                        (15, 82), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 100), 2)
        else:
            state        = "CAPTURING"
            count        = 0
            last_capture = time.time()

    # ======================================================
    # CAPTURING
    # ======================================================
    elif state == "CAPTURING":
        now = time.time()

        if now - last_capture >= DELAY_ANTAR:
            fname = os.path.join(SAVE_DIR, label_now, f"{label_now}_{count+1:03d}.jpg")
            cv2.imwrite(fname, frame)
            count       += 1
            last_capture = now
            print(f"  Tersimpan [{label_now}] {count}/{TOTAL_GAMBAR}")

        if count >= TOTAL_GAMBAR:
            state = "DONE"

        panel(frame, 0, 0, W, 105)
        cv2.putText(frame, f"Merekam : {label_now.upper()}",
                    (15, 42), cv2.FONT_HERSHEY_DUPLEX, 0.9, (0, 255, 100), 2)
        cv2.putText(frame, f"Foto : {count} / {TOTAL_GAMBAR}",
                    (15, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 220, 255), 2)

        # Progress bar
        bx, by, bw, bh = 15, 90, W-30, 10
        cv2.rectangle(frame, (bx, by), (bx+bw, by+bh), (50, 50, 50), -1)
        fill = int(bw * count / TOTAL_GAMBAR)
        if fill > 0:
            cv2.rectangle(frame, (bx, by), (bx+fill, by+bh), (0, 220, 120), -1)
        cv2.rectangle(frame, (bx, by), (bx+bw, by+bh), (150, 150, 150), 1)

        # Indikator REC kedip
        if int(now * 3) % 2 == 0:
            cv2.circle(frame, (W-30, 30), 10, (0, 0, 255), -1)
            cv2.putText(frame, "REC", (W-68, 37),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 2)

    # ======================================================
    # DONE
    # ======================================================
    elif state == "DONE":
        panel(frame, W//2-220, H//2-70, 440, 140)
        cv2.putText(frame, "SELESAI !",
                    (W//2-90, H//2-25), cv2.FONT_HERSHEY_DUPLEX, 1.3, (0, 255, 150), 3)
        cv2.putText(frame, f"{TOTAL_GAMBAR} foto [{label_now}] tersimpan",
                    (W//2-195, H//2+15), cv2.FONT_HERSHEY_SIMPLEX, 0.68, (200, 200, 200), 1)
        cv2.putText(frame, "SPACE = label berikutnya   Q = keluar",
                    (W//2-195, H//2+48), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255, 255, 100), 1)

    cv2.imshow("Ambil Dataset Kata", frame)

    # ======================================================
    # KEYBOARD
    # ======================================================
    key = cv2.waitKey(1)

    # tekan q atau ESC untuk keluar
    if key == ord('q') or key == 27:
        print("Program ditutup...")
        break

    # Tombol 1-9 untuk label 1-9, tombol 0 untuk label ke-10
    if key == ord('n'):
        if label_idx < len(LABEL_KATA)-1:
            label_idx += 1
            state = "MENU"
    
    if key == ord('p'):
        if label_idx > 0:
            label_idx -= 1
            state = "MENU"

    if key == ord(' '):
        if state == "MENU":
            ex = len([f for f in os.listdir(os.path.join(SAVE_DIR, label_now)) if f.endswith(".jpg")])
            if ex >= TOTAL_GAMBAR:
                print(f"  [{label_now}] sudah penuh ({ex} foto), pindah label berikutnya.")
                if label_idx < len(LABEL_KATA) - 1:
                    label_idx += 1
            else:
                state           = "COUNTDOWN"
                countdown_start = time.time()

        elif state == "DONE":
            if label_idx < len(LABEL_KATA) - 1:
                label_idx += 1
                state      = "MENU"
            else:
                print("\n Semua label selesai!")
                break

cap.release()
cv2.destroyAllWindows()
print(f"\n Dataset tersimpan di : {os.path.abspath(SAVE_DIR)}/")


  [salam kenal] sudah penuh (70 foto), pindah label berikutnya.
  [berasal] sudah penuh (70 foto), pindah label berikutnya.
  [berpikir] sudah penuh (70 foto), pindah label berikutnya.
  [makan] sudah penuh (70 foto), pindah label berikutnya.
  [mandi] sudah penuh (70 foto), pindah label berikutnya.
  [saya] sudah penuh (70 foto), pindah label berikutnya.
  [tidur] sudah penuh (70 foto), pindah label berikutnya.
  [nama] sudah penuh (70 foto), pindah label berikutnya.
  [maaf] sudah penuh (70 foto), pindah label berikutnya.
  [tolong] sudah penuh (70 foto), pindah label berikutnya.
  [mulai] sudah penuh (70 foto), pindah label berikutnya.
  [sementara] sudah penuh (70 foto), pindah label berikutnya.
  Tersimpan [tunggu] 1/70
  Tersimpan [tunggu] 2/70
  Tersimpan [tunggu] 3/70
  Tersimpan [tunggu] 4/70
  Tersimpan [tunggu] 5/70
  Tersimpan [tunggu] 6/70
  Tersimpan [tunggu] 7/70
  Tersimpan [tunggu] 8/70
  Tersimpan [tunggu] 9/70
  Tersimpan [tunggu] 10/70
  Tersimpan [tunggu] 11/70
  T

In [2]:
# Ambil Kata model squence
# =========================================================
# IMPORT LIBRARY
# =========================================================
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os
import time

# =========================================================
# MEDIAPIPE
# =========================================================
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,  # 2 tangan
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

# =========================================================
# LABEL KATA
# =========================================================
LABEL_KATA = [
    "salam kenal",
    "butuh",
    "berpikir",
    "makan",
    "mandi",
    "saya",
    "tidur",
    "tunggu",
    "maaf",
    "tolong",
    "berasal",
]

# =========================================================
# KONFIGURASI
# =========================================================
SEQUENCE_LENGTH = 20
JUMLAH_DATA_PER_KATA = 70

OUTPUT_FILE = "bahasa_kata2.csv"

SAVE_DELAY = 2

# =========================================================
# HEADER CSV
# =========================================================
header = []

for frame in range(SEQUENCE_LENGTH):

    # 42 landmark total (21 kiri + 21 kanan)
    for landmark in range(42):

        header += [
            f"f{frame}_x{landmark}",
            f"f{frame}_y{landmark}",
            f"f{frame}_z{landmark}"
        ]

header.append("label")

# =========================================================
# BUAT FILE CSV
# =========================================================
if not os.path.exists(OUTPUT_FILE):
    pd.DataFrame(columns=header).to_csv(OUTPUT_FILE, index=False)

# =========================================================
# CAMERA
# =========================================================
cap = cv2.VideoCapture(0)

# =========================================================
# VARIABEL
# =========================================================
sequence_data = []

index_kata = 0
jumlah_data = 0

recording = False

# =========================================================
# INFO
# =========================================================
print("\n====================================")
print(" DATASET BAHASA ISYARAT 2 TANGAN")
print("====================================")
print("N = pindah kata")
print("Q = keluar")
print("====================================\n")

# =========================================================
# LOOP
# =========================================================
while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    result = hands.process(rgb)

    current_label = LABEL_KATA[index_kata]

    # =====================================================
    # KATA BERIKUTNYA
    # =====================================================
    if index_kata + 1 < len(LABEL_KATA):
        next_label = LABEL_KATA[index_kata + 1]
    else:
        next_label = "SELESAI"

    # =====================================================
    # DETEKSI 2 TANGAN
    # =====================================================
    frame_landmarks = []

    left_hand = None
    right_hand = None

    # -----------------------------------------
    # CEK TANGAN
    # -----------------------------------------
    if result.multi_hand_landmarks and result.multi_handedness:

        for hand_landmarks, handedness in zip(
            result.multi_hand_landmarks,
            result.multi_handedness
        ):

            label = handedness.classification[0].label

            # Gambar landmark
            mp_draw.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS
            )

            # Simpan kiri / kanan
            if label == "Left":
                left_hand = hand_landmarks

            elif label == "Right":
                right_hand = hand_landmarks

    # =====================================================
    # TANGAN KIRI
    # =====================================================
    if left_hand:

        for lm in left_hand.landmark:
            frame_landmarks.extend([lm.x, lm.y, lm.z])

    else:
        # Isi nol jika tidak ada tangan kiri
        frame_landmarks.extend([0.0] * 63)

    # =====================================================
    # TANGAN KANAN
    # =====================================================
    if right_hand:

        for lm in right_hand.landmark:
            frame_landmarks.extend([lm.x, lm.y, lm.z])

    else:
        # Isi nol jika tidak ada tangan kanan
        frame_landmarks.extend([0.0] * 63)

    # =====================================================
    # SIMPAN FRAME KE SEQUENCE
    # =====================================================
    if len(frame_landmarks) == 126:

        sequence_data.append(frame_landmarks)

        # Maksimal 20 frame
        if len(sequence_data) > SEQUENCE_LENGTH:
            sequence_data.pop(0)

    # =====================================================
    # AUTO SAVE
    # =====================================================
    current_time = time.time()

    if (
        recording
        and len(sequence_data) == SEQUENCE_LENGTH
        and jumlah_data < JUMLAH_DATA_PER_KATA
        and current_time - last_save_time > SAVE_DELAY
    ):

        # Flatten sequence
        flattened = np.array(sequence_data).flatten()

        # Tambah label
        data_final = list(flattened)
        data_final.append(current_label)

        # Simpan ke CSV
        df = pd.DataFrame([data_final])

        df.to_csv(
            OUTPUT_FILE,
            mode='a',
            header=False,
            index=False
        )

        jumlah_data += 1

        print(f"[INFO] {current_label} -> data ke-{jumlah_data} tersimpan")

        # Reset timer
        last_save_time = current_time

        # Reset sequence
        sequence_data = []

    # =====================================================
    # UI
    # =====================================================

    # Kata sekarang
    cv2.putText(
        frame,
        f"Kata : {current_label}",
        (10, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (0,255,0),
        2
    )

    # Kata berikutnya
    cv2.putText(
        frame,
        f"Berikut : {next_label}",
        (10, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255,200,0),
        2
    )

    # Jumlah data
    cv2.putText(
        frame,
        f"Data : {jumlah_data}/{JUMLAH_DATA_PER_KATA}",
        (10, 120),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0,255,255),
        2
    )

    # Frame
    cv2.putText(
        frame,
        f"Frame : {len(sequence_data)}/{SEQUENCE_LENGTH}",
        (10, 160),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255,255,0),
        2
    )

    # Kata ke
    cv2.putText(
        frame,
        f"Kata ke : {index_kata+1}/{len(LABEL_KATA)}",
        (10, 200),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255,0,255),
        2
    )

    # =====================================================
    # STATUS RECORDING
    # =====================================================
    if not recording:
    
        cv2.putText(
            frame,
            "TEKAN SPACE UNTUK MULAI",
            (10, 240),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0,255,0),
            3
        )
    
    else:
    
        cv2.putText(
            frame,
            "RECORDING...",
            (10, 240),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0,0,255),
            3
        )

    # =====================================================
    # STATUS SELESAI
    # =====================================================
    if jumlah_data >= JUMLAH_DATA_PER_KATA:

        cv2.putText(
            frame,
            "DATA KATA SELESAI",
            (10, 260),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0,0,255),
            3
        )

        cv2.putText(
            frame,
            "TEKAN N UNTUK LANJUT",
            (10, 300),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0,0,255),
            3
        )

    # =====================================================
    # TAMPILKAN
    # =====================================================
    cv2.imshow("Dataset Bahasa Isyarat 2 Tangan", frame)

    key = cv2.waitKey(1)

    # =====================================================
    # MULAI RECORD
    # =====================================================
    if key == 32:  # SPACE
    
        recording = not recording
    
        sequence_data = []
    
        if recording:
            print("\n[INFO] RECORDING DIMULAI")
        else:
            print("\n[INFO] RECORDING DIHENTIKAN")
            
    # =====================================================
    # PINDAH KATA
    # =====================================================
    if key == ord('n'):

        index_kata += 1

        if index_kata >= len(LABEL_KATA):

            print("\nSEMUA DATA SELESAI!")
            break

        jumlah_data = 0
        sequence_data = []

        print(f"\n[INFO] Pindah ke kata: {LABEL_KATA[index_kata]}")

    # =====================================================
    # KELUAR
    # =====================================================
    elif key == ord('q'):
        break

# =========================================================
# RELEASE
# =========================================================
cap.release()
cv2.destroyAllWindows()

print("\nDataset selesai dibuat!")


 DATASET BAHASA ISYARAT 2 TANGAN
N = pindah kata
Q = keluar


[INFO] RECORDING DIMULAI
[INFO] salam kenal -> data ke-1 tersimpan
[INFO] salam kenal -> data ke-2 tersimpan
[INFO] salam kenal -> data ke-3 tersimpan
[INFO] salam kenal -> data ke-4 tersimpan
[INFO] salam kenal -> data ke-5 tersimpan
[INFO] salam kenal -> data ke-6 tersimpan
[INFO] salam kenal -> data ke-7 tersimpan
[INFO] salam kenal -> data ke-8 tersimpan
[INFO] salam kenal -> data ke-9 tersimpan
[INFO] salam kenal -> data ke-10 tersimpan
[INFO] salam kenal -> data ke-11 tersimpan
[INFO] salam kenal -> data ke-12 tersimpan
[INFO] salam kenal -> data ke-13 tersimpan
[INFO] salam kenal -> data ke-14 tersimpan
[INFO] salam kenal -> data ke-15 tersimpan
[INFO] salam kenal -> data ke-16 tersimpan
[INFO] salam kenal -> data ke-17 tersimpan
[INFO] salam kenal -> data ke-18 tersimpan
[INFO] salam kenal -> data ke-19 tersimpan
[INFO] salam kenal -> data ke-20 tersimpan
[INFO] salam kenal -> data ke-21 tersimpan
[INFO] salam kenal

In [7]:
# ambil kata satu tangan
# =========================================================
# IMPORT LIBRARY
# =========================================================
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os
import time

# =========================================================
# MEDIAPIPE
# =========================================================
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

# =========================================================
# LABEL KATA
# =========================================================
LABEL_KATA = [
    "tunggu",
]

# =========================================================
# KONFIGURASI
# =========================================================
SEQUENCE_LENGTH = 20
JUMLAH_DATA_PER_KATA = 70

OUTPUT_FILE = "bahasa_kata.csv"

SAVE_DELAY = 2

# =========================================================
# HEADER CSV
# =========================================================
header = []

for frame in range(SEQUENCE_LENGTH):
    for landmark in range(21):
        header += [
            f"f{frame}_x{landmark}",
            f"f{frame}_y{landmark}",
            f"f{frame}_z{landmark}"
        ]

header.append("label")

if not os.path.exists(OUTPUT_FILE):
    pd.DataFrame(columns=header).to_csv(OUTPUT_FILE, index=False)

# =========================================================
# KAMERA
# =========================================================
cap = cv2.VideoCapture(0)

# =========================================================
# VARIABEL
# =========================================================
sequence_data = []

index_kata = 0
jumlah_data = 0

last_save_time = time.time()

print("\n================================")
print("AUTO SAVE AKTIF")
print("N = Pindah kata")
print("Q = Keluar")
print("================================\n")

# =========================================================
# LOOP
# =========================================================
while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    result = hands.process(rgb)

    current_label = LABEL_KATA[index_kata]

    # =====================================================
    # KATA BERIKUTNYA
    # =====================================================
    if index_kata + 1 < len(LABEL_KATA):
        next_label = LABEL_KATA[index_kata + 1]
    else:
        next_label = "SELESAI"

    # =====================================================
    # DETEKSI TANGAN
    # =====================================================
    if result.multi_hand_landmarks:

        for hand_landmarks in result.multi_hand_landmarks:

            mp_draw.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS
            )

            landmarks = []

            for lm in hand_landmarks.landmark:
                landmarks.extend([lm.x, lm.y, lm.z])

            if len(landmarks) == 63:

                sequence_data.append(landmarks)

                # Simpan maksimal 20 frame
                if len(sequence_data) > SEQUENCE_LENGTH:
                    sequence_data.pop(0)

    # =====================================================
    # AUTO SAVE
    # =====================================================
    current_time = time.time()

    if (
        len(sequence_data) == SEQUENCE_LENGTH
        and jumlah_data < JUMLAH_DATA_PER_KATA
        and current_time - last_save_time > SAVE_DELAY
    ):

        flattened = np.array(sequence_data).flatten()

        data_final = list(flattened)
        data_final.append(current_label)

        df = pd.DataFrame([data_final])

        df.to_csv(
            OUTPUT_FILE,
            mode='a',
            header=False,
            index=False
        )

        jumlah_data += 1

        print(f"[INFO] {current_label} -> data ke-{jumlah_data} tersimpan")

        last_save_time = current_time

        sequence_data = []

    # =====================================================
    # TAMPILKAN INFO
    # =====================================================

    # Kata sekarang
    cv2.putText(
        frame,
        f"Kata Sekarang : {current_label}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

    # Kata berikutnya
    cv2.putText(
        frame,
        f"Berikutnya : {next_label}",
        (10, 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 200, 0),
        2
    )

    # Jumlah data
    cv2.putText(
        frame,
        f"Data : {jumlah_data}/{JUMLAH_DATA_PER_KATA}",
        (10, 110),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 255),
        2
    )

    # Frame
    cv2.putText(
        frame,
        f"Frame : {len(sequence_data)}/{SEQUENCE_LENGTH}",
        (10, 150),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 0),
        2
    )

    # Kata ke berapa
    cv2.putText(
        frame,
        f"Kata ke : {index_kata+1}/{len(LABEL_KATA)}",
        (10, 190),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 0, 255),
        2
    )

    # =====================================================
    # STATUS PINDAH
    # =====================================================
    if jumlah_data >= JUMLAH_DATA_PER_KATA:

        cv2.putText(
            frame,
            "DATA KATA INI SELESAI",
            (10, 240),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 0, 255),
            3
        )

        cv2.putText(
            frame,
            "TEKAN N UNTUK LANJUT",
            (10, 280),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 0, 255),
            3
        )

    # =====================================================
    # TAMPILKAN
    # =====================================================
    cv2.imshow("Dataset Bahasa Isyarat", frame)

    key = cv2.waitKey(1)

    # =====================================================
    # PINDAH KATA
    # =====================================================
    if key == ord('n'):

        index_kata += 1

        if index_kata >= len(LABEL_KATA):

            print("\nSEMUA DATA SELESAI!")
            break

        jumlah_data = 0
        sequence_data = []

        print(f"\n[INFO] Pindah ke kata: {LABEL_KATA[index_kata]}")

    # =====================================================
    # KELUAR
    # =====================================================
    elif key == ord('q'):
        break

# =========================================================
# RELEASE
# =========================================================
cap.release()
cv2.destroyAllWindows()

print("\nDataset selesai dibuat!")


AUTO SAVE AKTIF
N = Pindah kata
Q = Keluar

[INFO] tunggu -> data ke-1 tersimpan
[INFO] tunggu -> data ke-2 tersimpan
[INFO] tunggu -> data ke-3 tersimpan
[INFO] tunggu -> data ke-4 tersimpan
[INFO] tunggu -> data ke-5 tersimpan
[INFO] tunggu -> data ke-6 tersimpan
[INFO] tunggu -> data ke-7 tersimpan
[INFO] tunggu -> data ke-8 tersimpan
[INFO] tunggu -> data ke-9 tersimpan
[INFO] tunggu -> data ke-10 tersimpan
[INFO] tunggu -> data ke-11 tersimpan
[INFO] tunggu -> data ke-12 tersimpan
[INFO] tunggu -> data ke-13 tersimpan
[INFO] tunggu -> data ke-14 tersimpan
[INFO] tunggu -> data ke-15 tersimpan
[INFO] tunggu -> data ke-16 tersimpan
[INFO] tunggu -> data ke-17 tersimpan
[INFO] tunggu -> data ke-18 tersimpan
[INFO] tunggu -> data ke-19 tersimpan
[INFO] tunggu -> data ke-20 tersimpan
[INFO] tunggu -> data ke-21 tersimpan
[INFO] tunggu -> data ke-22 tersimpan
[INFO] tunggu -> data ke-23 tersimpan
[INFO] tunggu -> data ke-24 tersimpan
[INFO] tunggu -> data ke-25 tersimpan
[INFO] tunggu 

In [3]:
import pandas as pd

# =========================================================
# FILE CSV
# =========================================================
CSV_FILE = "bahasa_kata.csv"

# =========================================================
# BACA CSV
# =========================================================
df = pd.read_csv(CSV_FILE)

print("Jumlah data sebelum dihapus:")
print(df["label"].value_counts())

# =========================================================
# HAPUS LABEL "berasal"
# =========================================================
df = df[df["label"] != "berasal"]

# =========================================================
# SIMPAN KEMBALI
# =========================================================
df.to_csv(CSV_FILE, index=False)

print("\nLabel 'berasal' berhasil dihapus!")

print("\nJumlah data setelah dihapus:")
print(df["label"].value_counts())

Jumlah data sebelum dihapus:
label
salam kenal    140
berasal         72
berpikir        70
makan           70
mandi           70
saya            70
tidur           70
nama            70
maaf            70
tolong          70
Name: count, dtype: int64

Label 'berasal' berhasil dihapus!

Jumlah data setelah dihapus:
label
salam kenal    140
berpikir        70
makan           70
mandi           70
saya            70
tidur           70
nama            70
maaf            70
tolong          70
Name: count, dtype: int64
